# TrustFed-RL — Full D1 IoMT matrix on Google Colab

**Updated 13 Aug 2026** · governance `colleague_v1` · seeds **42–53** · **30** rounds

Runs the **entire** master matrix on Colab:

**B0 → B1 → B3 → B4 → B5−S → B5 → B5† → B4†**

Use **this notebook only**. Do not use `Run_IoMT_Experiments.ipynb`.

Laptop and Drive are separate: you may leave a local job running; treat **Colab/Drive metrics** as the paper results when this finishes.

Guide: [`COLAB_INSTRUCTIONS.md`](COLAB_INSTRUCTIONS.md) · packer: `bash scripts/pack_colab_handoff.sh`


## 0) Laptop prep

```bash
cd TrustFed-Agent
bash scripts/pack_colab_handoff.sh
```

Upload to **`My Drive/trustfed/`**:

| File | Full from scratch | Resume / skip finished |
|------|-------------------|------------------------|
| `TrustFed-Agent-colab.zip` | yes | yes |
| `iomt_data_bundle.zip` | yes | yes |
| `iomt_metrics_latest.zip` | **no** (delete on Drive if present) | yes |
| this notebook | *File → Upload notebook* | same |

**Full from scratch:** do not upload the metrics zip (or delete `iomt_metrics_latest.zip` on Drive first).

Colab runtime: **CPU**. Multi-day run — keep tab open or use Colab Pro.


## 1) Mount Drive + install code from zip


In [ ]:
from google.colab import drive
import os, shutil, zipfile
from pathlib import Path

drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/trustfed")
DRIVE.mkdir(parents=True, exist_ok=True)
ROOT = Path("/content/TrustFed-Agent")

def find_run_experiments(base: Path):
    hits = list(base.rglob("run_experiments.py"))
    return hits[0].parent if hits else None

repo_zip = next((p for p in [DRIVE / "TrustFed-Agent-colab.zip", DRIVE / "TrustFed-Agent.zip"] if p.exists()), None)
assert repo_zip is not None, "Upload TrustFed-Agent-colab.zip to My Drive/trustfed/"

# Always refresh code from Drive zip
if ROOT.exists():
    shutil.rmtree(ROOT)

print(f"Extracting {repo_zip} ...")
tmp = Path("/content/_tf_extract")
if tmp.exists():
    shutil.rmtree(tmp)
tmp.mkdir()
with zipfile.ZipFile(repo_zip, "r") as zf:
    zf.extractall(tmp)
src = find_run_experiments(tmp)
assert src is not None, "run_experiments.py not found in zip"
shutil.move(str(src), str(ROOT))
shutil.rmtree(tmp, ignore_errors=True)

os.chdir(ROOT)
gov = ROOT / "data" / "governance" / "colleague_seed_v1.json"
print("OK:", ROOT)
print("Governance seed:", "FOUND" if gov.exists() else "MISSING")
assert gov.exists(), "Re-pack with pack_colab_handoff.sh"
print("Data zip on Drive:", (DRIVE / "iomt_data_bundle.zip").exists())
print("Metrics zip on Drive (skip seeds if present):", (DRIVE / "iomt_metrics_latest.zip").exists())


## 2) Install dependencies (NumPy 1.x)


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

%cd /content/TrustFed-Agent
!pip -q install -U pip
!pip -q install -r requirements.txt

import numpy, sklearn, gymnasium, stable_baselines3, torch
print("numpy", numpy.__version__, "| torch", torch.__version__)
assert numpy.__version__.startswith("1."), "Need NumPy 1.x"


## 3) Load IoMT data + confirm 30 rounds / seeds 42–53


In [ ]:
%cd /content/TrustFed-Agent
import json, zipfile, subprocess, sys
from pathlib import Path

cfg = json.loads(Path("config/experiment_config.json").read_text())
print("num_rounds =", cfg["model"]["num_rounds"])
print("seeds =", cfg["seeds"])
assert cfg["model"]["num_rounds"] >= 30
assert list(cfg["seeds"]) == list(range(42, 54))

ROOT = Path(".").resolve()
DRIVE = Path("/content/drive/MyDrive/trustfed")
clients = ROOT / "data" / "CSVs" / "iomt_clients"
n = len(list(clients.glob("*.csv"))) if clients.exists() else 0
print("local hospitals:", n)
if n < 12:
    bundle = DRIVE / "iomt_data_bundle.zip"
    assert bundle.exists(), "Upload iomt_data_bundle.zip to My Drive/trustfed/"
    print("Extracting", bundle)
    with zipfile.ZipFile(bundle) as zf:
        zf.extractall(ROOT)
subprocess.check_call([sys.executable, "scripts/check_data.py"])


## 4) Optional: wipe Colab metrics for a true from-scratch run

Run this **only** for mode A (full redo). Skips if you want resume from Drive.


In [ ]:
%cd /content/TrustFed-Agent
from pathlib import Path
import shutil

# Set True for a clean full matrix on this Colab runtime (does not delete the Drive zip by itself)
WIPE_LOCAL_METRICS = True

metrics = Path("results/trustfed_agent/metrics")
if WIPE_LOCAL_METRICS and metrics.exists():
    shutil.rmtree(metrics)
    print("Wiped local Colab metrics dir")
metrics.mkdir(parents=True, exist_ok=True)

drive_zip = Path("/content/drive/MyDrive/trustfed/iomt_metrics_latest.zip")
print("Drive metrics zip present:", drive_zip.exists())
print("For from-scratch: delete that Drive zip in the Drive UI (or rename it) BEFORE section 5,")
print("otherwise the runner will restore and skip finished seeds.")


## 5) Run the **full** master matrix

Use **`--no-mount`**: Drive is already mounted in section 1.  
Calling `drive.mount()` again inside `!python` fails with  
`AttributeError: 'NoneType' object has no attribute 'kernel'` (no IPython kernel in the subprocess).

```bash
python scripts/colab_run_experiments.py --task trustfed_rl_matrix --no-mount --download
```

Backs up to Drive after every seed. If Colab disconnects, **re-run this cell**.  
Do **not** use `sed` to patch the runner — `--no-mount` is the supported fix.


In [ ]:
%cd /content/TrustFed-Agent
!python scripts/colab_run_experiments.py --task trustfed_rl_matrix --no-mount --download


### Optional subsets (always keep `--no-mount` after section 1)

```bash
# Only B5† + B4†
# !python scripts/colab_run_experiments.py --task trustfed_rl_matrix --phases B5star,B4star --no-mount --download

# Remaining after local B0–B5-S
# !python scripts/colab_run_experiments.py --task trustfed_rl_matrix --phases B5,B5star,B4star --no-mount --download

# Optional B5-P (weighted update hiding; does not overwrite D1 B5 JSON)
# !python scripts/colab_run_experiments.py --task trustfed_rl_matrix --phases B5P --no-mount --download
```


## 6) Status + analyze


In [ ]:
%cd /content/TrustFed-Agent
!python scripts/check_trustfed_rl_iomt_status.py
!python run_experiments.py analyze
print("Drive metrics: /content/drive/MyDrive/trustfed/iomt_metrics_latest.zip")
